# 05 - Long-stay Prediction (LOS > 24 h)
Leakage-free features use karke.

In [ ]:
import pandas as pd, json
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, classification_report
df = pd.read_csv('../data/processed/event_log_clean.csv')
ca = pd.read_csv('../data/raw/case_attributes.csv')

In [ ]:
X = pd.get_dummies(ca[['age','gender','triage_level','admission_type','pathway','n_events']], drop_first=True)
y = (ca.los_hours > 24).astype(int)
Xtr,Xte,ytr,yte = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
print('train', Xtr.shape, 'test', Xte.shape, 'long-stay share', round(y.mean()*100,2),'%')

In [ ]:
m = GradientBoostingClassifier(random_state=42).fit(Xtr,ytr)
p = m.predict_proba(Xte)[:,1]
print('AUC:', round(roc_auc_score(yte,p),4))
print(classification_report(yte,(p>0.5).astype(int), zero_division=0))

In [ ]:
import joblib, pathlib
pathlib.Path('../outputs/models').mkdir(parents=True, exist_ok=True)
joblib.dump(m, '../outputs/models/best_model.joblib'); print('model saved')